# Do penny-stock explosions have a tradable conditional edge?

### A veto-filter study of exchange-listed US microcaps, 2016-2026

---

## The question, stated precisely

The published evidence against penny-stock day trading is strong and consistent.
Eraker & Ready (2015) find OTC returns are negative on average and violently
right-skewed; the SEC's own outcomes study puts the median retail OTC holding
return at **-11.9%**; Barber, Lee, Liu & Odean find **fewer than 1%** of day
traders earn predictably positive abnormal returns net of fees.

All of that measures **unconditional** expectation. That is the gap this study
examines. A filter with high precision on a *toxic sub-population* can move
$E[r \mid \text{survived}]$ even when $E[r]$ is deeply negative -- and several of
those same papers effectively hand over the filter conditions.

So the hypothesis under test is:

> **H1.** There exists a veto set $V$, computable from free data at 09:35 ET with
> no look-ahead, such that
> $$E[r_{\text{net}} \mid \neg V] > 0$$
> with a confidence interval excluding zero after multiple-testing correction, at
> a retained population fraction and capacity worth trading.

**H0** is that no such set exists. The honest prior favours H0, and this notebook
is built so rejecting H0 requires surviving every check below -- while *failing*
to reject it is a cheap, publishable result rather than a wasted month.


## Why a veto and not a predictor

The asymmetry is the whole idea.

Predicting *which* microcap will explode is close to hopeless: the outcome is
dominated by a handful of enormous winners in a fat right tail, and by
information (promotion schedules, offering timing) that is not public before the
fact. Recognising which candidates are **structurally poisoned** is far easier,
and the literature supplies most of the conditions.

Formally, partition the candidate population $C$ into a vetoed part $V$ and its
complement. Then

$$E[r \mid C] = \pi\, E[r \mid V] + (1-\pi)\, E[r \mid \neg V]$$

where $\pi = P(V)$. Rearranged:

$$E[r \mid \neg V] = \frac{E[r \mid C] - \pi\, E[r \mid V]}{1 - \pi}$$

**Lemma 1 (a veto need not predict winners).** If $E[r \mid V] < E[r \mid C]$, then
$E[r \mid \neg V] > E[r \mid C]$ for any $\pi \in (0,1)$.

*Proof.* Immediate from the identity above: subtracting a below-average mass and
renormalising raises the mean. $\square$

The practical content: a veto only has to identify a sub-population that is
*worse than average*. It never has to identify winners. That is a far weaker
requirement, and it is why this study is worth running despite the literature.

**Lemma 2 (improvement is not profitability).** Lemma 1 places no lower bound on
$E[r \mid \neg V]$. If $E[r \mid C] \ll 0$, a veto can raise the conditional mean
substantially and still leave it negative.

This is not pedantry -- for penny stocks the baseline *is* strongly negative, so
the two questions come apart in exactly the regime we are in. The notebook
therefore reports **both**: a test against zero (profitable?) and a paired test
against the unfiltered baseline (does the veto help?).


## Design of the experiment

### Population (the denominator)

A result is meaningless without saying what it is a statement *about*. The
candidate screen defines the population, and every condition is knowable at
09:30 ET:

| Condition | Value | Knowable at 09:30? |
|---|---|---|
| Opening gap | $\geq +10\%$ vs prior close | yes -- uses the opening print |
| Price | \$0.50-\$10.00, **previous unadjusted** close | yes |
| Liquidity | trailing 20-day dollar volume $\geq$ \$100k | yes -- shifted |
| History | $\geq 30$ prior bars | yes |

### Timeline

```
09:30 - 09:34   observation window   features may use ONLY these bars
09:35           decision + fill      entry pays spread + impact
09:36 - 15:55   barrier scan         first barrier touched wins
15:55           vertical barrier     forced exit
```

### Labels: triple-barrier on a minute path

Following Lopez de Prado (2018, ch. 3), adapted from daily series to minute
bars. Five assumptions, each **pessimistic by construction**:

1. **Ambiguous minutes resolve against the trade.** If a bar's high crosses the
   target *and* its low crosses the stop, the bar cannot say which came first --
   we assume the stop. Microcap minute ranges are enormous, so the optimistic
   choice would manufacture edge out of bar granularity alone.
2. **No print means no fill.** Measured directly: one \$3 name printed 52 of 391
   minutes.
3. **Stops slip, targets do not.** A triggered stop is a market order crossing a
   falling book; a target is a resting limit.
4. **Entry pays the spread** plus square-root impact.
5. **Data gaps are suspicion, not absence** -- flagged `halt_suspect`, because
   free halt history does not exist and a halted position cannot be exited.

One caveat stated plainly: the scan starts at 09:36, so a barrier touched inside
the entry bar is missed. This is the only place the pessimism is not
one-directional.


In [ ]:
import pickle, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'research').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from research.penny import viz
from research.penny.eval import stats as pstats

pio.renderers.default = 'notebook'
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

with open(ROOT / 'data/research_penny/analysis.pkl', 'rb') as fh:
    A = pickle.load(fh)

print(f"trades      : {A['n_trades']:,}")
print(f"period      : {A['date_min']} .. {A['date_max']}")
print(f"holdout     : final {A['holdout_frac']:.0%} of sessions, sealed")


## Validity checks come first

A leakage bug does not announce itself -- it produces a **better** number, so
nothing looks wrong. "The result improved" is therefore the least reliable signal
available, and these tests run before any performance figure is read.

Each deliberately destroys the thing being measured and asserts the edge dies
with it. An edge that survives a destroyed label is an edge in the *pipeline*.

| Test | What it breaks | Expected |
|---|---|---|
| `constant_return` | all returns identical | no veto can create dispersion |
| `shuffled_labels` | feature-outcome association | FDR survivors $\approx q$, not $\gg q$ |
| `date_shift` | feature-outcome pairing, keeping day clustering | same |

> **Note on `shuffled_labels`.** This test initially *failed*, and the reason is
> instructive. The search tests each subset against zero -- correct for the real
> study. Under a shuffle every subset inherits the panel's overall mean, which for
> penny stocks is emphatically non-zero, so every subset read as "significant".
> Returns are now demeaned before the null search, which poses the question the
> null actually cares about: can the vetoes still *discriminate*?


In [ ]:
nulls = A.get('nulls')
if nulls is not None:
    display(nulls[['name', 'status', 'observed', 'reference', 'detail']])
    if nulls['passed'].all():
        print('\nAll null tests PASS -- results below are interpretable.')
    else:
        print('\nNULL TESTS FAILED -- do not interpret anything below.')


## Cross-validation design

Random K-fold is invalid here for three separate reasons, any one fatal:

1. **Temporal order.** A fold using 2026 to predict 2022 is not a forecast.
2. **Same-day clustering.** Trades in one session share a market factor. Splitting
   a day across train and test lets the model learn that day's regime from one
   half and "predict" the other. Folds are split on **whole days**.
3. **Serial correlation.** Adjacent days are not independent, hence an **embargo**.

**A distinction worth making explicitly.** Labels here are same-session, so label
horizons do **not** overlap and the classic purge for overlapping labels is
unnecessary. The embargo addresses serial correlation, not "feature-window
bleed" -- in a strictly sequential design a test day's trailing 60-day window
reaching into the train *period* is not leakage, it is exactly what live trading
does. No embargo of any length changes that. (An earlier version of this
docstring claimed otherwise and was wrong; it matters because converting to
Lopez de Prado-style purged K-fold, where train may *follow* test, would make the
requirement genuinely different.)


In [ ]:
for line in A['folds']:
    print(line)


## The candidate vetoes

Every veto is registered with the claim it encodes and its source. A veto with
no prior is a fishing expedition dressed as a hypothesis -- and it would still
inflate the multiple-testing correction for everything else.

One veto is marked **diagnostic**: `veto_delisted_soon` uses future information
by construction. It is never included in a tradable search (the search function
raises if you try); it exists only to bound how much of any edge is simply the
avoidance of terminal names -- which a live system would need filings to detect.


In [ ]:
display(A['veto_registry'])


## Baseline: what the unfiltered population earns

This is the denominator every later claim is measured against.


In [ ]:
b = A['baseline']
print(f'trades           {b.n_trades:,} over {b.n_days:,} sessions')
print(f'E[net return]    {b.mean_net:+.4%}   95% CI [{b.ci_lo:+.4%}, {b.ci_hi:+.4%}]')
print(f'median           {b.median_net:+.4%}')
print(f'hit rate         {b.hit_rate:.1%}')
print(f'P(target)        {b.p_target:.1%}    P(stop) {b.p_stop:.1%}')
print(f'Sharpe           {b.sharpe:.2f}')
print(f'max drawdown     {b.max_dd:.1%}')


### Where price actually travels

Before asking whether a filter helps, check the barriers are reachable at all.
A profit target beyond the bulk of the maximum-favourable-excursion mass is
unreachable **by construction**, no matter how good the signal.


In [ ]:
trades = A['trades_sample']
bar = A['params']['barriers']
viz.excursion_density(trades, bar['profit_target'], bar['stop_loss']).show()


## Effect of each veto in isolation

Diagnostic only. Reading the best row off this table and calling it a result is
precisely the error the fold machinery exists to prevent -- these are in-sample
marginals over the tuning period.


In [ ]:
display(A['single'].style.format({
    'retained_frac': '{:.1%}', 'mean_kept': '{:+.3%}', 'ci_lo_kept': '{:+.3%}',
    'ci_hi_kept': '{:+.3%}', 'mean_removed': '{:+.3%}', 'lift': '{:+.3%}',
    'p_kept': '{:.3f}', 'sharpe_kept': '{:.2f}',
}))
viz.veto_lift(A['single']).show()


## The conditional expectation surface

The headline figure: the precision/recall trade-off expressed in **return space**.
Left means filtering harder; up means earning more per trade. H1 requires a point
above zero whose interval excludes zero, at an x-value worth trading.

**This is an in-sample search and is reported as such**, together with the number
of hypotheses examined. Under a pure null, the best of thousands of subsets looks
excellent -- which is why the out-of-sample section below, not this one, carries
the actual claim.


In [ ]:
table = A['search']
print(f"veto sets searched : {int(table['n_hypotheses'].iloc[0]):,}")
print(f"passing BH-FDR     : {int(table['q_pass'].sum())}")
viz.conditional_surface(table, A['baseline'].mean_net).show()


In [ ]:
cols = ['name', 'n_trades', 'retained_frac', 'mean_net', 'ci_lo', 'ci_hi',
        'p_value', 'q_pass', 'sharpe', 'hit_rate']
display(table[cols].head(15).style.format({
    'retained_frac': '{:.1%}', 'mean_net': '{:+.3%}', 'ci_lo': '{:+.3%}',
    'ci_hi': '{:+.3%}', 'p_value': '{:.3f}', 'sharpe': '{:.2f}', 'hit_rate': '{:.1%}',
}))


### The fair baseline

A veto retaining $X\%$ must beat **random** filtering that retains $X\%$.
Otherwise the apparent improvement is just a smaller, noisier sample.

The placebo drops whole *days*, matching how real vetoes behave -- many veto
conditions are day-level facts, so a trade-level placebo would be an easier bar
to clear and would flatter the result.


In [ ]:
p = A.get('placebo', {})
best = A.get('best_in_sample')
if p and best is not None:
    print(f"best in-sample set   {best['name']}")
    print(f"  mean               {best['mean_net']:+.4%}")
    print(f"  placebo mean       {p['placebo_mean']:+.4%}")
    print(f"  placebo 5-95%      [{p['placebo_p05']:+.4%}, {p['placebo_p95']:+.4%}]")
    beats = best['mean_net'] > p['placebo_p95']
    print(f"  beats placebo p95  {beats}")

d = A.get('best_vs_baseline', {})
if d:
    print(f"\npaired vs baseline   {d['delta']:+.4%} "
          f"95% CI [{d['lo']:+.4%}, {d['hi']:+.4%}]  helps={d['helps']}")


## Out-of-sample: the only quotable number

Veto sets are selected on each fold's **training days** and scored on its
**held-out days**. Everything above is exploratory.

The gap between the two bars in the figure below is the most diagnostic quantity
in the study. A large, consistent gap is the signature of selection
overfitting -- the set was chosen because it looked good in training and did not
survive contact with held-out data.


In [ ]:
wf = A['wf_summary']
display(wf[['fold', 'test_start', 'test_end', 'chosen_vetoes', 'train_mean',
            'test_mean', 'test_n', 'test_retained', 'degradation']].style.format({
    'train_mean': '{:+.3%}', 'test_mean': '{:+.3%}',
    'test_retained': '{:.1%}', 'degradation': '{:+.3%}',
}))
viz.walk_forward_stability(wf).show()


In [ ]:
o = A.get('oos')
if o is not None:
    print(f'OUT-OF-SAMPLE (stitched across folds)')
    print(f'  trades         {o.n_trades:,} over {o.n_days:,} sessions')
    print(f'  E[net]         {o.mean_net:+.4%}   95% CI [{o.ci_lo:+.4%}, {o.ci_hi:+.4%}]')
    print(f'  retained       {o.retained_frac:.1%}')
    print(f'  hit rate       {o.hit_rate:.1%}')
    print(f'  Sharpe         {o.sharpe:.2f}')
    print(f'  Deflated SR    {A["oos_dsr"]:.3f}   (P that the Sharpe survives the search)')
    print(f'  max drawdown   {o.max_dd:.1%}')
    print()
    verdict = 'REJECTED' if o.ci_lo > 0 else 'NOT REJECTED'
    print(f'  H0 (no profitable veto set): {verdict}')


### Deflated Sharpe: pricing in the search

The expected maximum Sharpe under the null **grows with the number of strategies
tried**. Comparing an observed Sharpe against zero after searching thousands of
veto sets is the textbook backtest-overfitting error.

The Deflated Sharpe Ratio (Bailey & Lopez de Prado, 2014) compares it against
that expected maximum instead, and additionally corrects for skew and kurtosis --
which matter enormously here, since a handful of huge winners inflate the sample
Sharpe in a way the Gaussian formula does not penalise.


In [ ]:
curves = {'all candidates': A['baseline_curve']}
if 'oos_curve' in A and not A['oos_curve'].empty:
    curves['walk-forward OOS'] = A['oos_curve']
viz.equity_curves(curves).show()


In [ ]:
oos = A.get('oos_trades')
if oos is not None and not oos.empty:
    viz.return_distributions(
        A['trades_sample']['net_return'], oos['net_return'],
        filtered_label='after veto (out-of-sample)',
    ).show()


## Veto interactions

Vetoes are not independent -- `veto_thin_adv` and `veto_wide_spread` both proxy
illiquidity, so combining them adds little beyond either alone. The diagonal is
each veto in isolation; the midpoint of the colour scale is the no-veto baseline.


In [ ]:
from research.penny.features import veto as veto_mod
names = [v for v in veto_mod.veto_names(False)][:10]
viz.veto_interaction(A['search'], names, A['baseline'].mean_net).show()


In [ ]:
# Match on (symbol, date), not the index: oos_trades was concatenated with
# ignore_index=True, so its positional index would collide spuriously.
base = A['trades_sample'].copy()
oos = A.get('oos_trades')
if oos is not None and not oos.empty:
    kept = set(zip(oos['symbol'], oos['date']))
    base['population'] = [
        'after veto' if k in kept else 'all candidates'
        for k in zip(base['symbol'], base['date'])
    ]
    viz.outcome_mix(base, 'population').show()


## The sealed holdout

The final slice of sessions was removed **before any tuning** and is opened
exactly once, here. Nothing in the parameter search, the veto search, or the
walk-forward ever saw it.

This is the number that matters. Everything else was, in one way or another,
fitted.


In [ ]:
h, hb = A.get('holdout'), A.get('holdout_baseline')
if h is not None:
    print(f"veto set applied : {', '.join(A['holdout_vetoes']) or '(none)'}")
    print()
    print(f'{"":<18}{"baseline":>14}{"after veto":>14}')
    print(f'{"trades":<18}{hb.n_trades:>14,}{h.n_trades:>14,}')
    print(f'{"E[net]":<18}{hb.mean_net:>13.3%}{h.mean_net:>14.3%}')
    print(f'{"CI low":<18}{hb.ci_lo:>13.3%}{h.ci_lo:>14.3%}')
    print(f'{"CI high":<18}{hb.ci_hi:>13.3%}{h.ci_hi:>14.3%}')
    print(f'{"hit rate":<18}{hb.hit_rate:>13.1%}{h.hit_rate:>14.1%}')
    print(f'{"Sharpe":<18}{hb.sharpe:>14.2f}{h.sharpe:>14.2f}')
    print()
    if h.ci_lo > 0:
        print('Holdout interval excludes zero: H1 survives its final test.')
    else:
        print('Holdout interval includes zero: H0 is NOT rejected on unseen data.')


## Capacity

A percentage return is not a business. At 1% participation in the day's dollar
volume, the dollar profit per day bounds what this could ever be worth -- and in
microcaps that bound is the binding constraint, not the edge.

This is the mechanism Hou, Xue & Zhang (2020) identify: microcaps are ~3% of
market value but ~60% of listed names, and anomalies concentrate there precisely
*because* they cannot be traded away.


In [ ]:
oos = A.get('oos_trades')
if oos is not None and not oos.empty and 'adv_20' in oos.columns:
    cap = A['params']['barriers']['participation_cap']
    size = oos['adv_20'] * cap
    pnl = size * oos['net_return']
    per_day = pnl.groupby(oos['date']).sum()
    print(f'participation cap      {cap:.0%} of trailing 20-day dollar volume')
    print(f'median position size   ${size.median():,.0f}')
    print(f'median daily P&L       ${per_day.median():,.0f}')
    print(f'mean daily P&L         ${per_day.mean():,.0f}')
    print(f'annualised (252d)      ${per_day.mean() * 252:,.0f}')


## Conclusions

*(Filled from the run below -- see `research/penny/LOG.md` for the full process
record, including the bugs found and fixed along the way.)*

## Sources

**Method**

- Lopez de Prado, M. (2018). *Advances in Financial Machine Learning*, ch. 3
  (triple-barrier labelling), ch. 7 (purged K-fold, embargoing). Wiley.
- Bailey, D. & Lopez de Prado, M. (2014). "The Deflated Sharpe Ratio".
  *Journal of Portfolio Management* 40(5).
- Benjamini, Y. & Hochberg, Y. (1995). "Controlling the False Discovery Rate".
  *JRSS B* 57(1).
- Corwin, S. & Schultz, P. (2012). "A Simple Way to Estimate Bid-Ask Spreads
  from Daily High and Low Prices". *Journal of Finance* 67(2).

**The vetoes**

- Bali, T., Cakici, N. & Whitelaw, R. (2011). "Maxing Out: Stocks as Lotteries
  and the Cross-Section of Expected Returns". *JFE* 99(2). -- the MAX effect.
- Avramov, D., Chordia, T. & Goyal, A. (2006). "Liquidity and Autocorrelations
  in Individual Stock Returns". *Journal of Finance* 61(5).
- Frieder, L. & Zittrain, J. (2007). "Spam Works: Evidence from Stock Touts".
  *Hastings Comm. & Ent. Law Journal* 30(3).
- Beneish, M., Lee, C. & Nichols, C. (2015). "In Short Supply: Short Sellers and
  Stock Returns". *Journal of Accounting and Economics* 60(1).
- Hillion, P. & Vermaelen, T. (2004). "Death Spiral Convertibles". *JFE* 71(2).

**Why the baseline is what it is**

- Eraker, B. & Ready, M. (2015). "Do investors overpay for stocks with
  lottery-like payoffs?". *JFE* 115(3).
- White, J. (2016). "Outcomes of Investing in OTC Stocks". SEC DERA white paper.
- Hou, K., Xue, C. & Zhang, L. (2020). "Replicating Anomalies". *RFS* 33(5).
- Novy-Marx, R. & Velikov, M. (2016). "A Taxonomy of Anomalies and Their Trading
  Costs". *RFS* 29(1).
- Barber, B., Lee, Y., Liu, Y. & Odean, T. "Do Day Traders Rationally Learn About
  Their Ability?"
- Chague, F., De-Losso, R. & Giovannetti, B. (2020). "Day Trading for a Living?"
